In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [9]:
df_0 = pd.read_csv("../data/round_3/prices_round_3_day_0.csv", delimiter= ";")
df_1 = pd.read_csv("../data/round_3/prices_round_3_day_1.csv", delimiter= ";")
df_2 = pd.read_csv("../data/round_3/prices_round_3_day_2.csv", delimiter= ";")
df_0.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,0,0,VEV_5400,22,25,NaN,NaN,NaN,NaN,24,25,NaN,NaN,NaN,NaN,23.0,0.0
1,0,0,VEV_6500,0,16,NaN,NaN,NaN,NaN,1,16,NaN,NaN,NaN,NaN,0.5,0.0
2,0,0,VEV_5500,8,25,NaN,NaN,NaN,NaN,9,25,NaN,NaN,NaN,NaN,8.5,0.0
3,0,0,VEV_5200,100,19,NaN,NaN,NaN,NaN,103,6,104.0,13.0,NaN,NaN,101.5,0.0
4,0,0,VEV_5300,52,6,51.0,19.0,NaN,NaN,54,25,NaN,NaN,NaN,NaN,53.0,0.0


In [10]:
df_0.tail()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
119995,0,999900,VEV_5100,162,23,NaN,NaN,NaN,NaN,166,10,167.0,13.0,NaN,NaN,164.0,0.0
119996,0,999900,VEV_6000,0,21,NaN,NaN,NaN,NaN,1,21,NaN,NaN,NaN,NaN,0.5,0.0
119997,0,999900,HYDROGEL_PACK,9950,12,9947.0,21.0,NaN,NaN,9966,12,9968.0,21.0,NaN,NaN,9958.0,0.0
119998,0,999900,VELVETFRUIT_EXTRACT,5241,69,NaN,NaN,NaN,NaN,5247,69,NaN,NaN,NaN,NaN,5244.0,0.0
119999,0,999900,VEV_4500,736,10,734.0,13.0,NaN,NaN,752,10,754.0,13.0,NaN,NaN,744.0,0.0


In [6]:
# concat the three dataframes and make timestamp continuous across days
df = pd.concat([df_0, df_1, df_2], ignore_index=True)
df["timestamp"] = df["timestamp"] + df["day"] * 24 * 3600
df.shape

(360000, 17)

In [11]:
df.tail()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
359995,2,1172700,VELVETFRUIT_EXTRACT,5293,25,5292.0,39.0,NaN,NaN,5298,64,NaN,NaN,NaN,NaN,5295.5,0.0
359996,2,1172700,VEV_4500,787,10,785.0,17.0,NaN,NaN,804,10,806.0,17.0,NaN,NaN,795.5,0.0
359997,2,1172700,VEV_5400,19,28,NaN,NaN,NaN,NaN,21,28,NaN,NaN,NaN,NaN,20.0,0.0
359998,2,1172700,VEV_6500,0,11,NaN,NaN,NaN,NaN,1,11,NaN,NaN,NaN,NaN,0.5,0.0
359999,2,1172700,VEV_5500,6,28,NaN,NaN,NaN,NaN,8,28,NaN,NaN,NaN,NaN,7.0,0.0


In [14]:
import plotly.graph_objects as go
import pandas as pd

products = ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]

# Stitch days by offsetting timestamps
dfs = []
offset = 0
for day_num, df in enumerate([df_0, df_1, df_2]):
    tmp = df.copy()
    tmp["ts_global"] = tmp["timestamp"] + offset
    tmp["day"] = day_num
    dfs.append(tmp)
    offset += tmp["timestamp"].max() + 100  # +100 to avoid gap between days

combined = pd.concat(dfs, ignore_index=True)

fig = go.Figure()

colors = {"HYDROGEL_PACK": "royalblue", "VELVETFRUIT_EXTRACT": "tomato"}

for product in products:
    subset = combined[combined["product"] == product].sort_values("ts_global")
    fig.add_trace(go.Scatter(
        x=subset["ts_global"],
        y=subset["mid_price"],
        mode="lines",
        name=product,
        line=dict(color=colors[product]),
    ))

# Add vertical lines at day boundaries
for day_num, df in enumerate([df_0, df_1]):
    boundary = df["timestamp"].max() + 100
    if day_num == 0:
        x_pos = boundary
    else:
        x_pos = df_0["timestamp"].max() + 100 + boundary
    fig.add_vline(x=x_pos, line_dash="dash", line_color="gray", annotation_text=f"Day {day_num+1}")

fig.update_layout(
    title="HYDROGEL_PACK vs VELVETFRUIT_EXTRACT — Days 0-1-2 stitched",
    xaxis_title="timestamp (stitched)",
    yaxis_title="mid_price",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()


In [15]:
for day_num, df in enumerate([df_0, df_1, df_2]):
    hydrogel = df[df["product"] == "HYDROGEL_PACK"].set_index("timestamp")["mid_price"].pct_change()
    velvet   = df[df["product"] == "VELVETFRUIT_EXTRACT"].set_index("timestamp")["mid_price"].pct_change()
    
    aligned = pd.concat([hydrogel, velvet], axis=1, keys=["HYDROGEL", "VELVET"]).dropna()
    corr = aligned["HYDROGEL"].corr(aligned["VELVET"])
    print(f"Day {day_num} (returns): corr = {corr:.4f}  (n={len(aligned)})")


Day 0 (returns): corr = 0.0111  (n=9999)
Day 1 (returns): corr = 0.0120  (n=9999)
Day 2 (returns): corr = -0.0050  (n=9999)


In [26]:
products = ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"]
lags = range(-10_000, 10_000)
results = {}

for day_num, df in enumerate([df_0, df_1, df_2]):
    hydrogel = df[df["product"] == "HYDROGEL_PACK"].set_index("timestamp")["mid_price"].pct_change()
    velvet   = df[df["product"] == "VELVETFRUIT_EXTRACT"].set_index("timestamp")["mid_price"].pct_change()
    aligned  = pd.concat([hydrogel, velvet], axis=1, keys=["H", "V"]).dropna()

    day_corrs = {}
    for lag in lags:
        # lag > 0: HYDROGEL leads VELVET by `lag` ticks
        # lag < 0: VELVET leads HYDROGEL by `|lag|` ticks
        day_corrs[lag] = aligned["H"].corr(aligned["V"].shift(-lag))
    
    results[f"Day {day_num}"] = day_corrs

corr_df = pd.DataFrame(results, index=lags)
corr_df.index.name = "lag (ticks)"

# Style: background gradient so high correlations stand out
# styled = (
#     corr_df.style
#     .background_gradient(cmap="RdYlGn", axis=None, vmin=-1, vmax=1)
#     .format("{:.4f}")
#     .set_caption("Returns cross-correlation: HYDROGEL vs VELVETFRUIT (lag in ticks, +lag = HYDROGEL leads)")
# )
# styled


/Users/thibautdesauty/Documents/Prosperity4/IMC_PROSPERITY_4_COMPETITION/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/thibautdesauty/Documents/Prosperity4/IMC_PROSPERITY_4_COMPETITION/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/thibautdesauty/Documents/Prosperity4/IMC_PROSPERITY_4_COMPETITION/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/Users/thibautdesauty/Documents/Prosperity4/IMC_PROSPERITY_4_COMPETITION/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/thibautdesauty/Documents/Prosperity4/IMC_PROSPERITY_4_COMPETIT

In [27]:
fig = go.Figure()
for col in corr_df.columns:
    fig.add_trace(go.Scatter(x=corr_df.index, y=corr_df[col], mode="lines", name=col))

fig.update_layout(
    title="Cross-correlation vs lag",
    xaxis_title="lag (ticks)  [+ = HYDROGEL leads]",
    yaxis_title="correlation",
)
fig.show()


In [24]:
corr_df.idxmax()

Day 0   -9997
Day 1   -9997
Day 2    9997
dtype: int64

In [25]:
print(aligned.shape)  # e.g. (10000, 2) → max useful lag is ±9999
print(corr_df.isna().sum())  # should show ~10k NaNs per day


(9999, 2)
Day 0    20005
Day 1    20005
Day 2    20005
dtype: int64


In [ ]:
from statsmodels.tsa.stattools import coint, adfuller
import statsmodels.api as sm
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# ── 1. Cointegration test + OLS hedge ratio ──────────────────────────────────
rows = []
spreads = {}

for day_num, df in enumerate([df_0, df_1, df_2]):
    hydrogel = df[df["product"] == "HYDROGEL_PACK"].set_index("timestamp")["mid_price"]
    velvet   = df[df["product"] == "VELVETFRUIT_EXTRACT"].set_index("timestamp")["mid_price"]
    aligned  = pd.concat([hydrogel, velvet], axis=1, keys=["H", "V"]).dropna()

    # Engle-Granger test
    score, pvalue, crits = coint(aligned["H"], aligned["V"])

    # OLS: H = alpha + beta * V  →  spread = H - beta*V - alpha
    X = sm.add_constant(aligned["V"])
    model = sm.OLS(aligned["H"], X).fit()
    alpha, beta = model.params
    spread = aligned["H"] - beta * aligned["V"] - alpha

    # ADF on spread to confirm stationarity
    adf_stat, adf_p, *_ = adfuller(spread, autolag="AIC")

    spreads[day_num] = spread
    rows.append({
        "Day": day_num,
        "EG t-stat": round(score, 4),
        "EG p-value": round(pvalue, 6),
        "Cointegrated?": "YES" if pvalue < 0.05 else "NO",
        "beta (hedge ratio)": round(beta, 4),
        "alpha": round(alpha, 2),
        "ADF spread p-value": round(adf_p, 6),
        "Spread stationary?": "YES" if adf_p < 0.05 else "NO",
    })

summary = pd.DataFrame(rows).set_index("Day")
styled_summary = (
    summary.style
    .map(lambda v: "background-color: #90EE90" if v == "YES" else
                        "background-color: #FFB6C1" if v == "NO" else "",
              subset=["Cointegrated?", "Spread stationary?"])
    .format({"EG p-value": "{:.6f}", "ADF spread p-value": "{:.6f}"})
    .set_caption("Cointegration summary — HYDROGEL vs VELVETFRUIT")
)
display(styled_summary)

# ── 2. Visualization ─────────────────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Day {d} — price series (normalised)" for d in range(3)] +
                   [f"Day {d} — spread (H - β·V - α)" for d in range(3)],
    column_widths=[0.5, 0.5],
)

for day_num, df in enumerate([df_0, df_1, df_2]):
    hydrogel = df[df["product"] == "HYDROGEL_PACK"].set_index("timestamp")["mid_price"]
    velvet   = df[df["product"] == "VELVETFRUIT_EXTRACT"].set_index("timestamp")["mid_price"]
    aligned  = pd.concat([hydrogel, velvet], axis=1, keys=["H", "V"]).dropna()
    spread   = spreads[day_num]

    row = day_num + 1

    # Normalise to 0 so both series are on the same axis
    h_norm = aligned["H"] - aligned["H"].iloc[0]
    v_norm = aligned["V"] - aligned["V"].iloc[0]

    fig.add_trace(go.Scatter(x=aligned.index, y=h_norm, mode="lines",
                             name="HYDROGEL", line=dict(color="royalblue"),
                             showlegend=(day_num == 0)), row=row, col=1)
    fig.add_trace(go.Scatter(x=aligned.index, y=v_norm, mode="lines",
                             name="VELVETFRUIT", line=dict(color="tomato"),
                             showlegend=(day_num == 0)), row=row, col=1)

    # Spread + mean ± 2σ bands
    mu, sigma = spread.mean(), spread.std()
    fig.add_trace(go.Scatter(x=spread.index, y=spread, mode="lines",
                             name="Spread", line=dict(color="purple", width=1),
                             showlegend=False), row=row, col=2)
    for sign, label in [(1, "+2σ"), (-1, "−2σ")]:
        fig.add_hline(y=mu + sign * 2 * sigma, line_dash="dash",
                      line_color="gray", row=row, col=2,
                      annotation_text=label, annotation_position="right")
    fig.add_hline(y=mu, line_dash="dot", line_color="black", row=row, col=2)

fig.update_layout(height=900, title_text="Cointegration analysis — HYDROGEL vs VELVETFRUIT")
fig.show()


AttributeError: 'Styler' object has no attribute 'applymap'